In [8]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import time
from census import Census # This is new...

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

from requests.exceptions import ConnectTimeout, ReadTimeout, RequestException

In [9]:
current_date = "2025-12"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [10]:
end_use = "naics?get=ALL_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/exports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_exports"] = df["ALL_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_exports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_exports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [11]:
country_list[0] = ""

In [12]:
country_list.extend(["0003", "0020"])

In [13]:
country_list

['',
 '1220',
 '2010',
 '5700',
 '5880',
 '4120',
 '4280',
 '4210',
 '5800',
 '3510',
 '4279',
 '5590',
 '5830',
 '4231',
 '5820',
 '5330',
 '4419',
 '6021',
 '4759',
 '5200',
 '3370',
 '3010',
 '4700',
 '5570',
 '5170',
 '5081',
 '5490',
 '4890',
 '4190',
 '3330',
 '5520',
 '0003',
 '0020']

In [14]:
end_use = "hs?get=CTY_NAME,ALL_VAL_MO,E_COMMODITY,E_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/exports/" + end_use 
surl = surl + my_key + "&COMM_LVL=HS10"

# Build list of all months from 2013-01 to the latest available month (set in 'date' above)
start = dt.date(2013, 1, 1)
end = dt.datetime.strptime(current_date, "%Y-%m").date()

month_list = []
current = start
while current <= end:
    month_list.append(current.strftime("%Y-%m"))
    # Advance by one month
    if current.month == 12:
        current = dt.date(current.year + 1, 1, 1)
    else:
        current = dt.date(current.year, current.month + 1, 1)

print(f"Months to download: {month_list[0]} to {month_list[-1]} ({len(month_list)} months)")
print(f"Countries to download: {len(country_list)}")
print(f"Total files: {len(month_list) * len(country_list)}")

for xxx in country_list:
    
    country_label = "TOTAL" if xxx == "" else xxx
    
    for month in month_list:
        
        out_file = f".\\data\\exports-hs10\\{country_label}data-{month}.parquet"
        
        if os.path.exists(out_file):
            continue
        
        print(f"Downloading {country_label} for {month}")
        
        url = surl + "&time=" + month
        
        if xxx != "":
            url = url + "&CTY_CODE=" + xxx
        
        max_retries = 5
        retry_count = 0
        r = None
        
        while retry_count < max_retries:
            try:
                r = requests.get(url, timeout=30)
                
                if r.status_code == 200:
                    break
                else:
                    print(f"Request failed with status {r.status_code}, waiting 30 seconds...")
                    time.sleep(30)
                    retry_count += 1
                    
            except (ConnectTimeout, ReadTimeout) as e:
                retry_count += 1
                print(f"Connection timeout on attempt {retry_count}/{max_retries}: {type(e).__name__}")
                if retry_count < max_retries:
                    wait_time = 30 * (2 ** (retry_count - 1))  # Exponential backoff
                    print(f"Waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)
                else:
                    print(f"Max retries exceeded for {country_label} {month}, skipping...")
                    
            except RequestException as e:
                print(f"Request failed with error: {e}")
                retry_count += 1
                if retry_count < max_retries:
                    time.sleep(30)
        
        if r is None or r.status_code != 200:
            print(f"Failed to download {country_label} {month} after {max_retries} attempts, skipping...")
            continue
        
        try:
            data = r.json()
            if len(data) < 2:
                print(f"No data returned for {country_label} {month}, skipping...")
                continue
            
            foo = pd.DataFrame(data[1:])
            foo.columns = data[0]
            
            pq.write_table(pa.Table.from_pandas(foo), out_file)
            
        except Exception as e:
            print(f"Error processing {country_label} {month}: {e}")

Months to download: 2013-01 to 2025-12 (156 months)
Countries to download: 33
Total files: 5148
Connection timeout on attempt 1/5: ReadTimeout
Waiting 30 seconds before retry...
